# Silver Transactions (`pipelines/02_silver_transactions.ipynb`)

This notebook turns raw ingested Bronze records (`workspace.sentinel_bronze.transactions`) into two curated Silver tables:
- **`workspace.sentinel_silver.transactions`**: Clean, validated, unique transactions (`record_disposition == "accepted"`) with personal cardholder attributes removed.
- **`workspace.sentinel_silver.quarantine`**: Rejected or duplicate records (`record_disposition != "accepted"`) with raw strings, source lineage, and validation reasons preserved for investigation.

**Pipeline flow**: `read → convert types → build business signature → validate fields → count/rank duplicates → route dispositions → split & save → basic verification`

In [ ]:
# Configuration: imports, table names, explicit business signature columns, and output column lists
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "workspace.sentinel_bronze.transactions"
SILVER_SCHEMA = "workspace.sentinel_silver"
SILVER_TRANSACTIONS_TABLE = f"{SILVER_SCHEMA}.transactions"
SILVER_QUARANTINE_TABLE = f"{SILVER_SCHEMA}.quarantine"
MANIFEST_PATH = "/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest"

# Explicit list of all 21 Bronze business columns compared when checking duplicate transaction IDs.
# Lineage columns (source_file, source_row_index, replay_batch, _rescued_data, _ingested_file_path, _ingestion_timestamp)
# are excluded so duplicate detection compares business content rather than file metadata.
BUSINESS_SIGNATURE_COLUMNS = [
    "amt",
    "category",
    "cc_num",
    "city",
    "city_pop",
    "dob",
    "first",
    "gender",
    "job",
    "last",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "merchant",
    "state",
    "street",
    "trans_date_trans_time",
    "trans_num",
    "unix_time",
    "zip",
]

# Personal cardholder attributes excluded from accepted Silver transactions
EXCLUDED_PERSONAL_COLUMNS = {
    "first", "last", "gender", "street", "city", "state",
    "zip", "lat", "long", "city_pop", "job", "dob",
}

ACCEPTED_SILVER_COLUMNS = [
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

QUARANTINE_SILVER_COLUMNS = [
    "record_disposition", "duplicate_status", "duplicate_rank", "validation_reasons",
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "raw_trans_num", "raw_cc_num", "raw_merchant", "raw_amt",
    "raw_trans_date_trans_time", "unix_time", "_rescued_data",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

### Step 1: Read Bronze Transactions

We read `workspace.sentinel_bronze.transactions`, confirm `is_fraud` is absent, and display a compact summary of ingested batches.
- **Example**: Each Bronze row stores source CSV fields as `STRING` (e.g., `amt = "42.50"`, `trans_date_trans_time = "2019-01-01 00:01:15"`) alongside `replay_batch` (`DATE`) and ingestion metadata.

In [ ]:
# Step 1: Read Bronze transactions, confirm no fraud labels, and display batch summary
bronze_df = spark.table(BRONZE_TABLE)
assert "is_fraud" not in bronze_df.columns, "Fraud labels must not appear in Bronze"

bronze_input_count = bronze_df.count()
print(f"Read {bronze_input_count:,} Bronze transactions from {BRONZE_TABLE}.")

display(
    bronze_df.agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("replay_batch").alias("ingested_batches"),
        F.min("replay_batch").alias("earliest_batch"),
        F.max("replay_batch").alias("latest_batch"),
    )
)

### Step 2a: Normalize Identifiers & Convert Types (`TIMESTAMP_NTZ` and `DECIMAL`)

We copy raw strings (`raw_trans_num`, `raw_amt`, etc.) for quarantine inspection, trim and convert blank strings to `NULL` via `NULLIF(TRIM(...), '')`, parse `event_time` as `TIMESTAMP_NTZ`, and cast `amt` to both `DECIMAL(38,18)` (`_wide_amount`) and `DECIMAL(18,2)` (`amount`).
- **Why two decimal casts?**: `TRY_CAST('19.999' AS DECIMAL(18,2))` does not return `NULL`—it silently rounds to `20.00`. Comparing `_wide_amount` (`19.999`) against `amount` (`20.00`) lets Step 3 catch values that would change when rounded to 2 decimal places (`amount_precision_changed`) or negative sub-cent values like `"-0.004"` that round to `0.00`.
- **Privacy in samples**: When displaying sample rows, we mask `cc_num` to show only the last 4 digits (`****1234`).

In [ ]:
# Step 2a: Preserve raw strings, normalize blank identifiers to NULL, and cast timestamp/decimal types
typed_df = (
    bronze_df
    .withColumn("raw_trans_num", F.col("trans_num"))
    .withColumn("raw_cc_num", F.col("cc_num"))
    .withColumn("raw_merchant", F.col("merchant"))
    .withColumn("raw_amt", F.col("amt"))
    .withColumn("raw_trans_date_trans_time", F.col("trans_date_trans_time"))
    .withColumn("trans_num", F.expr("NULLIF(TRIM(trans_num), '')"))
    .withColumn("cc_num", F.expr("NULLIF(TRIM(cc_num), '')"))
    .withColumn("merchant", F.expr("NULLIF(TRIM(merchant), '')"))
    .withColumn("category", F.expr("NULLIF(TRIM(category), '')"))
    .withColumn("_normalized_event_time_str", F.expr("NULLIF(TRIM(trans_date_trans_time), '')"))
    .withColumn("event_time", F.expr("TRY_CAST(_normalized_event_time_str AS TIMESTAMP_NTZ)"))
    .withColumn("_normalized_amt_str", F.expr("NULLIF(TRIM(amt), '')"))
    .withColumn("_wide_amount", F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(38, 18))"))
    .withColumn("amount", F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(18, 2))"))
)

# Display a 5-row sample with masked card IDs
display(
    typed_df.select(
        "trans_num",
        F.concat(F.lit("****"), F.substring(F.col("cc_num"), -4, 4)).alias("masked_cc_num"),
        "merchant",
        "raw_amt",
        "_wide_amount",
        "amount",
        "event_time",
    ).limit(5)
)

### Step 2b: Build the Business-Record JSON Signature (`F.struct` and `F.to_json`)

To distinguish **identical** duplicate rows from **conflicting** duplicate rows sharing a `trans_num`, we build a JSON string (`_business_signature`) across the explicit `BUSINESS_SIGNATURE_COLUMNS`.
- **Why `{"ignoreNullFields": "false"}` in `F.to_json`?**: Each column is trimmed and blank-normalized (`NULLIF(TRIM(col), '')`), and `{"ignoreNullFields": "false"}` keeps `NULL` values inside the serialized JSON as `null` without needing a magic sentinel string like `"__NULL__"`.
- **Small example**: Two rows with `trans_num = "TX_1"` and identical business fields produce the exact same `_business_signature` string (`1` distinct signature), whereas two rows with `amt = "25.00"` vs `amt = "99.00"` (or `category = NULL` vs `category = "__NULL__"`) produce `2` distinct signatures.

In [ ]:
# Step 2b: Build deterministic JSON _business_signature across BUSINESS_SIGNATURE_COLUMNS
signature_fields = [
    F.expr(f"NULLIF(TRIM(`{col_name}`), '')").cast("string").alias(col_name)
    for col_name in BUSINESS_SIGNATURE_COLUMNS
]

signed_df = typed_df.withColumn(
    "_business_signature",
    F.to_json(F.struct(*signature_fields), {"ignoreNullFields": "false"}),
)

### Step 3: Validate Required Fields, Timestamps, Amounts & Rescued Data (`F.array` and `F.filter`)

A single record can fail more than one check at once. Instead of stopping at the first error with `CASE WHEN`, we put every field check inside `F.array(...)` and remove `NULL`s with `F.filter(..., lambda r: r.isNotNull())`.
- **Identifier checks**: `missing_transaction_id`, `missing_card_id`, `missing_merchant`.
- **Timestamp checks**: `missing_event_time` (blank string) vs. `unparseable_event_time` (non-blank string that failed `TRY_CAST`).
- **Amount checks**: `missing_amount`, `unparseable_or_out_of_range_amount`, `amount_precision_changed` (`_wide_amount != amount`), and `non_positive_amount` (`_wide_amount <= 0`).
- **Rescued data check**: `unexpected_rescued_data` when `_rescued_data` is non-null and non-blank.
- **Small example**: `amt = "-0.004"` has `_wide_amount = -0.004` and `amount = 0.00`, so it records both `["amount_precision_changed", "non_positive_amount"]`.

In [ ]:
# Step 3: Collect all applicable field-level validation failure reasons into an array
raw_validation_reasons = F.array(
    # Required identifiers
    F.when(F.col("trans_num").isNull(), F.lit("missing_transaction_id")),
    F.when(F.col("cc_num").isNull(), F.lit("missing_card_id")),
    F.when(F.col("merchant").isNull(), F.lit("missing_merchant")),
    # Event timestamp
    F.when(F.col("_normalized_event_time_str").isNull(), F.lit("missing_event_time")),
    F.when(
        F.col("_normalized_event_time_str").isNotNull() & F.col("event_time").isNull(),
        F.lit("unparseable_event_time"),
    ),
    # Monetary amount
    F.when(F.col("_normalized_amt_str").isNull(), F.lit("missing_amount")),
    F.when(
        F.col("_normalized_amt_str").isNotNull()
        & (F.col("_wide_amount").isNull() | F.col("amount").isNull()),
        F.lit("unparseable_or_out_of_range_amount"),
    ),
    F.when(
        F.col("_wide_amount").isNotNull()
        & F.col("amount").isNotNull()
        & (F.col("_wide_amount") != F.col("amount").cast("decimal(38,18)")),
        F.lit("amount_precision_changed"),
    ),
    F.when(
        F.col("_wide_amount").isNotNull()
        & (F.col("_wide_amount") <= F.lit(0).cast("decimal(38,18)")),
        F.lit("non_positive_amount"),
    ),
    # Unexpected Auto Loader rescued data
    F.when(
        F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""),
        F.lit("unexpected_rescued_data"),
    ),
)

validated_df = signed_df.withColumn(
    "field_validation_reasons",
    F.filter(raw_validation_reasons, lambda reason: reason.isNotNull()),
)

display(
    validated_df.agg(
        F.count_if(F.size(F.col("field_validation_reasons")) == 0).alias("rows_passing_field_checks"),
        F.count_if(F.size(F.col("field_validation_reasons")) > 0).alias("rows_failing_field_checks"),
    )
)

### Step 4a: Count and Rank Duplicate Transaction IDs

Before filtering out invalid rows, we group by non-null `trans_num` across the entire Bronze input to compute:
- **`id_occurrence_count`**: How many rows share that `trans_num`.
- **`distinct_business_signatures`**: How many distinct `_business_signature` values exist for that `trans_num`.
- **`duplicate_rank`**: Deterministic `1, 2, ...` row number per `trans_num` ordered by ingestion timestamp, file path, and row index.
- **Small example**: If `TX_DUP` appears twice with identical fields, both rows get `id_occurrence_count = 2` and `distinct_business_signatures = 1`, with `duplicate_rank = 1` on the first copy and `duplicate_rank = 2` on the second copy.

In [ ]:
# Step 4a: Count occurrences and distinct signatures per trans_num and assign duplicate_rank
duplicate_summary = (
    validated_df.filter(F.col("trans_num").isNotNull())
    .groupBy("trans_num")
    .agg(
        F.count("*").alias("id_occurrence_count"),
        F.countDistinct("_business_signature").alias("distinct_business_signatures"),
    )
)

id_order_window = Window.partitionBy("trans_num").orderBy(
    F.col("_ingestion_timestamp").asc_nulls_last(),
    F.col("_ingested_file_path").asc_nulls_last(),
    F.expr("TRY_CAST(source_row_index AS BIGINT)").asc_nulls_last(),
    F.col("source_row_index").asc_nulls_last(),
)

ranked_df = (
    validated_df.join(duplicate_summary, on="trans_num", how="left")
    .withColumn(
        "duplicate_rank",
        F.when(F.col("trans_num").isNotNull(), F.row_number().over(id_order_window)).cast("int"),
    )
)

display(
    ranked_df.agg(
        F.count_if(F.col("id_occurrence_count") == 1).alias("unique_id_rows"),
        F.count_if((F.col("id_occurrence_count") > 1) & (F.col("distinct_business_signatures") == 1)).alias("identical_duplicate_rows"),
        F.count_if((F.col("id_occurrence_count") > 1) & (F.col("distinct_business_signatures") > 1)).alias("conflicting_duplicate_rows"),
        F.count_if(F.col("trans_num").isNull()).alias("missing_id_rows"),
    )
)

### Step 4b: Assign Duplicate Status & Route Record Disposition

Using the counts and rank from Step 4a, we assign `duplicate_status`, append any duplicate validation reason (`conflicting_duplicate_transaction_id` or `duplicate_identical_transaction_id`), and route every row to exactly one `record_disposition`:
- **`accepted`**: `size(validation_reasons) == 0` (valid fields and either a `unique_id` or `identical_duplicate_primary`).
- **`duplicate_identical`**: `duplicate_status == "identical_duplicate_extra"` with valid fields (`size(field_validation_reasons) == 0`).
- **`quarantined`**: Any row with field validation errors or a `conflicting_duplicate` ID (we never pick an arbitrary winner when copies conflict).

In [ ]:
# Step 4b: Assign duplicate_status, combine validation_reasons, and route record_disposition
classified_df = (
    ranked_df
    .withColumn(
        "duplicate_status",
        F.when(F.col("trans_num").isNull(), F.lit("missing_id"))
        .when(F.col("id_occurrence_count") == 1, F.lit("unique_id"))
        .when(
            (F.col("id_occurrence_count") > 1) & (F.col("distinct_business_signatures") > 1),
            F.lit("conflicting_duplicate"),
        )
        .when(
            (F.col("id_occurrence_count") > 1)
            & (F.col("distinct_business_signatures") == 1)
            & (F.col("duplicate_rank") == 1),
            F.lit("identical_duplicate_primary"),
        )
        .otherwise(F.lit("identical_duplicate_extra")),
    )
    .withColumn(
        "duplicate_validation_reasons",
        F.when(
            F.col("duplicate_status") == "conflicting_duplicate",
            F.array(F.lit("conflicting_duplicate_transaction_id")),
        )
        .when(
            F.col("duplicate_status") == "identical_duplicate_extra",
            F.array(F.lit("duplicate_identical_transaction_id")),
        )
        .otherwise(F.array().cast("array<string>")),
    )
    .withColumn(
        "validation_reasons",
        F.concat(F.col("field_validation_reasons"), F.col("duplicate_validation_reasons")),
    )
    .withColumn(
        "record_disposition",
        F.when(F.size(F.col("validation_reasons")) == 0, F.lit("accepted"))
        .when(
            (F.col("duplicate_status") == "identical_duplicate_extra")
            & (F.size(F.col("field_validation_reasons")) == 0),
            F.lit("duplicate_identical"),
        )
        .otherwise(F.lit("quarantined")),
    )
)

display(
    classified_df.groupBy("record_disposition", "duplicate_status")
    .agg(F.count("*").alias("row_count"))
    .orderBy("record_disposition", "duplicate_status")
)

### Step 5: Split Accepted and Quarantine Outputs

We split `classified_df` into two target DataFrames and verify that every Bronze input row is accounted for:
- **`accepted_silver_df`**: `record_disposition == "accepted"` with `ACCEPTED_SILVER_COLUMNS` (personal cardholder fields excluded).
- **`quarantine_silver_df`**: `record_disposition != "accepted"` (both `quarantined` and `duplicate_identical`) with `QUARANTINE_SILVER_COLUMNS`.

In [ ]:
# Step 5: Split classified rows into accepted Silver and quarantine projections
assert EXCLUDED_PERSONAL_COLUMNS.isdisjoint(ACCEPTED_SILVER_COLUMNS), (
    "Personal attributes must not appear in accepted Silver columns"
)

accepted_silver_df = (
    classified_df.filter(F.col("record_disposition") == "accepted")
    .select(*ACCEPTED_SILVER_COLUMNS)
)

quarantine_silver_df = (
    classified_df.filter(F.col("record_disposition") != "accepted")
    .select(*QUARANTINE_SILVER_COLUMNS)
)

disposition_counts = classified_df.agg(
    F.count("*").alias("classified_rows"),
    F.count_if(F.col("record_disposition") == "accepted").alias("accepted_rows"),
    F.count_if(F.col("record_disposition") == "quarantined").alias("quarantined_rows"),
    F.count_if(F.col("record_disposition") == "duplicate_identical").alias("duplicate_identical_rows"),
).collect()[0]

assert disposition_counts["classified_rows"] == bronze_input_count, "Row count mismatch after classification"
assert (
    disposition_counts["accepted_rows"]
    + disposition_counts["quarantined_rows"]
    + disposition_counts["duplicate_identical_rows"]
    == bronze_input_count
), "Disposition counts do not sum to Bronze input count"

print(
    f"Split {bronze_input_count:,} Bronze rows -> "
    f"{disposition_counts['accepted_rows']:,} accepted, "
    f"{disposition_counts['quarantined_rows']:,} quarantined, "
    f"{disposition_counts['duplicate_identical_rows']:,} duplicate_identical."
)

### Step 6: Save Silver and Quarantine Tables (`overwrite` Mode)

We rebuild both `workspace.sentinel_silver.transactions` and `workspace.sentinel_silver.quarantine` from the complete current Bronze snapshot using `.mode("overwrite")`.
- **Why full recomputation**: If a later batch introduces a conflicting duplicate `trans_num`, reclassifying the full Bronze snapshot removes the earlier record from `transactions` and routes all occurrences to `quarantine`.
- **Note**: Each `saveAsTable` is a separate Delta transaction; always run both this cell and Step 7 verification to completion.

In [ ]:
# Step 6: Overwrite workspace.sentinel_silver.transactions and workspace.sentinel_silver.quarantine
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

accepted_silver_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(SILVER_TRANSACTIONS_TABLE)

quarantine_silver_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(SILVER_QUARANTINE_TABLE)

print(f"Saved {SILVER_TRANSACTIONS_TABLE} and {SILVER_QUARANTINE_TABLE} in full-snapshot overwrite mode.")

### Step 7: Basic Silver & Quarantine Verification

We read both persisted tables back from Delta storage and verify:
1. Column lists match `ACCEPTED_SILVER_COLUMNS` and `QUARANTINE_SILVER_COLUMNS` (with `is_fraud` and personal columns excluded).
2. Total and daily batch counts (`saved_accepted + saved_quarantine`) reconcile with Bronze and `/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest`.
3. Accepted `trans_num` values are non-null and unique.

In [ ]:
# Step 7: Read back saved Silver tables and verify counts, manifest reconciliation, and unique accepted IDs
saved_silver_df = spark.table(SILVER_TRANSACTIONS_TABLE)
saved_quarantine_df = spark.table(SILVER_QUARANTINE_TABLE)
manifest_df = spark.read.parquet(MANIFEST_PATH)

assert saved_silver_df.columns == ACCEPTED_SILVER_COLUMNS, "Unexpected columns in Silver transactions"
assert saved_quarantine_df.columns == QUARANTINE_SILVER_COLUMNS, "Unexpected columns in Silver quarantine"
assert "is_fraud" not in saved_silver_df.columns and "is_fraud" not in saved_quarantine_df.columns

saved_accepted_count = saved_silver_df.count()
saved_quarantine_count = saved_quarantine_df.count()
expected_quarantine_count = (
    disposition_counts["quarantined_rows"] + disposition_counts["duplicate_identical_rows"]
)

assert saved_accepted_count == disposition_counts["accepted_rows"], "Saved accepted count mismatch"
assert saved_quarantine_count == expected_quarantine_count, "Saved quarantine count mismatch"
assert saved_accepted_count + saved_quarantine_count == bronze_input_count, "Total saved rows != Bronze count"

null_accepted_ids = saved_silver_df.filter(F.col("trans_num").isNull()).count()
distinct_accepted_ids = saved_silver_df.select("trans_num").distinct().count()
assert null_accepted_ids == 0, "NULL trans_num found in accepted Silver"
assert distinct_accepted_ids == saved_accepted_count, "Duplicate trans_num found in accepted Silver"

# Reconcile daily counts against the replay manifest for all ingested batches.
accepted_by_batch = saved_silver_df.groupBy("replay_batch").agg(
    F.count("*").alias("saved_accepted_rows"),
    F.countDistinct("trans_num").alias("distinct_accepted_ids"),
)
quarantine_by_batch = saved_quarantine_df.groupBy("replay_batch").agg(
    F.count("*").alias("saved_quarantine_rows"),
)
ingested_batches = bronze_df.select("replay_batch").distinct()

batch_reconciliation = (
    manifest_df.join(ingested_batches, on="replay_batch", how="inner")
    .join(accepted_by_batch, on="replay_batch", how="left")
    .join(quarantine_by_batch, on="replay_batch", how="left")
    .fillna(0, subset=["saved_accepted_rows", "distinct_accepted_ids", "saved_quarantine_rows"])
    .withColumn("saved_total_rows", F.col("saved_accepted_rows") + F.col("saved_quarantine_rows"))
    .withColumn("matches_manifest", F.col("saved_total_rows") == F.col("expected_transaction_count"))
    .orderBy("replay_batch")
)

assert batch_reconciliation.filter(~F.col("matches_manifest")).count() == 0, (
    "Daily Silver + quarantine counts do not match the manifest"
)

print(
    f"Verified {SILVER_TRANSACTIONS_TABLE}: {saved_accepted_count:,} rows "
    f"({distinct_accepted_ids:,} unique trans_num)."
)
print(f"Verified {SILVER_QUARANTINE_TABLE}: {saved_quarantine_count:,} rows.")
display(batch_reconciliation)